# 최신 단일채널 화자 구분 ASR 비교

Nemotron-3-Diarization + Qwen3-ASR-1.7B, VibeVoice-ASR 9B, MOSS-Transcribe-Diarize 0.9B를 같은 평가셋에 실행합니다. 실제 입력은 **항상 마이크 하나**이며, proposed와 같은 기준 채널(활성 마이크 중 최소 번호)을 선택합니다.

먼저 `LIMIT=2`로 추론·재개·채점을 확인한 뒤 `LIMIT=0`으로 전체 평가하세요. A100을 권장하며, 특히 VibeVoice는 가중치만 약 18GB이므로 T4에 적합하지 않습니다. 이 노트북은 모델을 학습하지 않습니다.

세 기본 설정은 20초 전체를 관측합니다. `full_audio` 참고 비교로 보고하며, 실시간 proposed와 관측 범위를 함께 표시합니다. 우위 여부는 실행 결과로 판단합니다.


In [ ]:
BASELINES: list[str] = ["nemotron", "vibevoice", "moss"]
LIMIT: int = 2  # @param {type:"integer"}
MAX_NEW_TOKENS: int = 2048  # @param {type:"integer"}
BRANCH: str = "main"  # @param {type:"string"}
COMMIT: str = ""  # @param {type:"string"}
REPO_URL: str = "https://github.com/hd1534/variable-channel-spatial-asr.git"
REPO_DIR: str = "/content/variable-channel-spatial-asr"
DRIVE_ROOT: str = "/content/drive/MyDrive/vcsasr"
ENV_ROOT: str = "/content/vcsasr-envs"
SETS: list[str] = ["eval-t2", "eval-t2-static"]
RUN_TAG: str = ""  # @param {type:"string"}
ASR_CACHE_NAME: str = "asr_cache"  # @param {type:"string"}


## 저장소와 Drive 준비

학습 노트북과 같은 Drive 경로를 사용하면 Nemotron과 proposed의 Qwen 전사 캐시를 공유합니다. 모델별 환경은 런타임 로컬 디스크에 설치하고, 추론 결과는 Drive에 매 세션 저장합니다. 같은 출력 파일에는 한 번에 하나의 실행만 쓰세요.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path


def run_command(arguments: list[str]) -> None:
    """Run a command without shell interpolation and stop on failure."""
    print("$", " ".join(arguments), flush=True)
    subprocess.run(arguments, check=True)


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
if not Path(REPO_DIR, ".git").is_dir():
    run_command(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                 REPO_URL, REPO_DIR])
run_command(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/*", "!/data/eval-dev/audio/"])
run_command(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", COMMIT or BRANCH])
run_command(["git", "-C", REPO_DIR, "checkout", "FETCH_HEAD"])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
run_command(["git", "log", "-1", "--format=%h %s"])
Path(DRIVE_ROOT, "mono_baselines").mkdir(parents=True, exist_ok=True)

os.environ["VCSASR_MODEL_CACHE"] = str(Path(DRIVE_ROOT, "models", "asr"))


def run_name(name: str) -> str:
    """Select a separate output directory when experiment settings change."""
    return f"{name}-{RUN_TAG}" if RUN_TAG else name


## 모델별 환경과 GPU 확인

기존 학습 환경을 갱신하지 않고 별도 venv에 공식 구현을 설치합니다. 모델 가중치와 공식 소스는 코드에 기록된 revision을 사용합니다. 설치된 패키지 목록도 결과 폴더에 보존합니다. 설치와 GPU 모델 실행은 Colab에서 확인해야 합니다.


In [ ]:
import shutil
import torch
from setup_asr_env import setup

if not torch.cuda.is_available():
    raise RuntimeError("런타임 유형을 GPU로 바꾼 뒤 다시 실행하세요.")
print(torch.cuda.get_device_name(0), "| VRAM GB", torch.cuda.get_device_properties(0).total_memory / 1e9)
if LIMIT < 0 or MAX_NEW_TOKENS < 1:
    raise ValueError("LIMIT은 0 이상, MAX_NEW_TOKENS는 1 이상이어야 합니다.")
if not BASELINES or any(name not in ("nemotron", "vibevoice", "moss") for name in BASELINES):
    raise ValueError("BASELINES를 확인하세요.")
profiles: list[str] = list(dict.fromkeys(BASELINES + (["qwen"] if "nemotron" in BASELINES else [])))
interpreters: dict[str, str] = {name: str(setup(name, Path(ENV_ROOT))) for name in profiles}
for name in profiles:
    destination: Path = Path(DRIVE_ROOT, "mono_baselines", "runtimes", name)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in ("resolved-requirements.txt", "vcsasr-runtime.json"):
        shutil.copy2(Path(ENV_ROOT, name, filename), destination / filename)


## 추론과 채점

완료한 세션은 건너뜁니다. 모델·디코딩 설정·오디오가 달라지면 기존 결과와 혼합하지 않고 중단하므로, 변경된 실험은 다른 출력 경로를 사용하세요. `LIMIT=2` 실행 후 `LIMIT=0`으로 바꾸면 나머지 세션을 이어서 처리합니다. 파싱 실패나 토큰 상한에 도달한 출력은 성공으로 저장하지 않습니다.

모델·토큰 상한을 바꿀 때에는 `RUN_TAG`로 새 실험 경로를 지정합니다. CPU/GPU 정밀도나 패키지 버전이 달라졌다면 `ASR_CACHE_NAME`도 새 이름으로 설정하고 학습 노트북과 맞추세요.


In [ ]:
for name in BASELINES:
    arguments: list[str] = [sys.executable, "mono_baselines.py", "--backend", name,
                            "--python", interpreters[name], "--out", str(Path(DRIVE_ROOT, "mono_baselines", run_name(name))),
                            "--asr-cache", str(Path(DRIVE_ROOT, ASR_CACHE_NAME)), "--device", "cuda",
                            "--limit", str(LIMIT), "--max-new-tokens", str(MAX_NEW_TOKENS), "--sets", *SETS]
    if name == "nemotron":
        arguments += ["--qwen-python", interpreters["qwen"]]
    run_command(arguments)


## 비교 결과

`paired M`은 상대 proposed가 사용한 마이크 수입니다. 아래 모든 baseline의 실제 입력은 **1ch**입니다. 같은 케이스끼리 SpkAcc와 cpWER를 비교하며, 동일 마이크의 출력을 서로 다른 M에서 반복 평가한 값을 독립 표본으로 간주하지 않습니다. 추후 신뢰구간을 계산할 때에는 세션 단위로 묶으세요.

Nemotron은 proposed와 같은 Qwen 전사·최소 1초 slot-run 규칙을 사용하고, VibeVoice/MOSS는 자체 전사를 사용합니다. 2인 평가 계약에 맞춰 예측 화자 수가 2명을 넘으면 발화 시간이 긴 두 화자를 유지하고 나머지는 최장 화자에 합칩니다. 이 후처리는 정답을 사용하지 않으며 결과에 정책을 기록합니다.


In [ ]:
reports: dict[str, dict] = {}
for name in BASELINES:
    report_path: Path = Path(DRIVE_ROOT, "mono_baselines", run_name(name), "score.json")
    reports[name] = json.loads(report_path.read_text(encoding="utf-8"))
for set_name in SETS:
    print(f"\n{set_name} | input=1ch | full_audio | LIMIT={LIMIT} (0=전체)")
    for name, report in reports.items():
        values: list[str] = []
        for m, metrics in report[set_name]["by_comparison_M"].items():
            values.append(f"paired M={m}: Acc {metrics['speaker_attribution_accuracy']:.3f}, cpWER {metrics['cpwer']:.3f}")
        print(report["baseline"]["run"], " | ".join(values))
print("===== VC-SASR MONO BASELINES =====")
print(json.dumps(reports, ensure_ascii=False))
print("===== END =====")


## Proposed와 함께 보기

학습 노트북에서 생성한 결과가 있으면 같은 Drive에서 비교표를 만듭니다. `input`은 실제 입력 채널 수이고 `scope`는 관측 범위입니다. 다른 평가 부분집합은 별도 population으로 표시합니다.


In [ ]:
run_command([sys.executable, "compare_asr.py", "--eval-root", str(Path(DRIVE_ROOT, "eval")),
             "--mono-root", str(Path(DRIVE_ROOT, "mono_baselines")),
             "--out", str(Path(DRIVE_ROOT, "asr_comparison.json")), "--sets", *SETS])
